# 10. Bounding Box와 IoU

이 노트북은 `09_분류에서_객체_탐지로.ipynb` 다음 단계로, 객체 탐지에서 가장 기본이 되는 **bounding box 표현 방식**, **IoU(Intersection over Union)**, **confidence score**를 다룹니다.

객체 탐지는 단순히 `무엇인가?`를 맞히는 문제가 아니라 `무엇이 어디에 있는가?`를 맞히는 문제입니다. 따라서 모델의 출력에는 클래스뿐 아니라 위치 정보가 반드시 포함됩니다.

이번 노트북의 목표는 다음과 같습니다.

- bounding box를 여러 좌표 형식으로 표현할 수 있습니다.
- `xyxy`, `xywh`, `cxcywh` 형식의 차이를 이해합니다.
- 두 박스가 얼마나 겹치는지 IoU로 계산합니다.
- confidence score가 탐지 결과에서 어떤 의미를 갖는지 이해합니다.


## 10-1. 준비

이번 노트북은 실제 딥러닝 모델을 학습하지 않습니다. 대신 좌표와 면적 계산을 직접 구현하면서 객체 탐지의 기본 수학을 확인합니다.


In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['axes.unicode_minus'] = False


## 10-2. Bounding box란 무엇인가?

Bounding box는 이미지 안의 객체를 감싸는 직사각형입니다. 객체의 정확한 외곽선을 모두 그리는 segmentation과 달리, detection에서는 보통 직사각형 하나로 객체 위치를 표현합니다.

예를 들어 고양이가 이미지의 왼쪽 위 근처에 있다면 다음과 같은 정보를 저장할 수 있습니다.

- 클래스: `cat`
- 박스 좌표: `(x1, y1, x2, y2)`

여기서 `x1, y1`은 왼쪽 위 꼭짓점, `x2, y2`는 오른쪽 아래 꼭짓점입니다.


In [ ]:
image_width, image_height = 240, 180
cat_box_xyxy = (45, 35, 145, 125)  # x1, y1, x2, y2

fig, ax = plt.subplots(figsize=(7, 5))
ax.set_xlim(0, image_width)
ax.set_ylim(image_height, 0)
ax.set_facecolor('#eef6ff')
ax.set_title('Bounding box 예시')
ax.set_xlabel('x')
ax.set_ylabel('y')

# 간단한 객체 모양과 bounding box를 함께 그립니다.
ax.add_patch(Rectangle((60, 50), 70, 60, color='#f5b76e', alpha=0.9))
x1, y1, x2, y2 = cat_box_xyxy
ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='crimson', linewidth=3))
ax.text(x1, y1 - 6, 'cat box', color='crimson', fontsize=12, weight='bold')
ax.scatter([x1, x2], [y1, y2], color='black')
ax.text(x1 + 3, y1 + 14, '(x1, y1)', fontsize=10)
ax.text(x2 - 55, y2 - 8, '(x2, y2)', fontsize=10)

plt.show()


## 10-3. 대표적인 박스 좌표 형식

같은 박스도 여러 방식으로 표현할 수 있습니다. 라이브러리나 논문마다 형식이 다르므로, 어떤 형식을 쓰는지 항상 확인해야 합니다.

- `xyxy`: `(x1, y1, x2, y2)` 왼쪽 위와 오른쪽 아래 꼭짓점
- `xywh`: `(x, y, w, h)` 왼쪽 위 좌표와 너비, 높이
- `cxcywh`: `(cx, cy, w, h)` 중심 좌표와 너비, 높이

YOLO 계열 모델은 보통 중심 좌표 기반 표현을 많이 사용합니다. 반면 시각화나 IoU 계산에서는 `xyxy` 형식이 편한 경우가 많습니다.


In [ ]:
def xyxy_to_xywh(box):
    x1, y1, x2, y2 = box
    return (x1, y1, x2 - x1, y2 - y1)


def xywh_to_xyxy(box):
    x, y, w, h = box
    return (x, y, x + w, y + h)


def xyxy_to_cxcywh(box):
    x1, y1, x2, y2 = box
    w = x2 - x1
    h = y2 - y1
    cx = x1 + w / 2
    cy = y1 + h / 2
    return (cx, cy, w, h)


def cxcywh_to_xyxy(box):
    cx, cy, w, h = box
    return (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)


box_xyxy = (45, 35, 145, 125)
box_xywh = xyxy_to_xywh(box_xyxy)
box_cxcywh = xyxy_to_cxcywh(box_xyxy)

print('xyxy  :', box_xyxy)
print('xywh  :', box_xywh)
print('cxcywh:', box_cxcywh)
print('cxcywh -> xyxy:', cxcywh_to_xyxy(box_cxcywh))


## 10-4. 정규화 좌표

실제 데이터셋에서는 좌표를 픽셀 값 그대로 저장하기도 하고, 이미지 크기로 나눈 정규화 좌표로 저장하기도 합니다.

예를 들어 이미지 크기가 `240 x 180`이고 박스의 중심 좌표가 `(95, 80)`이라면, 정규화 중심 좌표는 다음과 같습니다.

- `cx_norm = cx / image_width`
- `cy_norm = cy / image_height`

정규화 좌표는 이미지 크기가 달라도 항상 `0~1` 범위에 들어가기 때문에 모델 출력으로 다루기 좋습니다.


In [ ]:
def normalize_cxcywh(box, image_width, image_height):
    cx, cy, w, h = box
    return (cx / image_width, cy / image_height, w / image_width, h / image_height)


def denormalize_cxcywh(box, image_width, image_height):
    cx, cy, w, h = box
    return (cx * image_width, cy * image_height, w * image_width, h * image_height)


normalized_box = normalize_cxcywh(box_cxcywh, image_width, image_height)
restored_box = denormalize_cxcywh(normalized_box, image_width, image_height)

print('원본 cxcywh:', box_cxcywh)
print('정규화 좌표:', tuple(round(v, 3) for v in normalized_box))
print('복원 좌표  :', restored_box)


## 10-5. 박스 면적 계산

IoU를 계산하려면 먼저 박스의 면적을 계산할 수 있어야 합니다. `xyxy` 형식에서는 너비와 높이를 다음처럼 구합니다.

- 너비: `x2 - x1`
- 높이: `y2 - y1`
- 면적: `(x2 - x1) * (y2 - y1)`

단, 잘못된 박스가 들어와 너비나 높이가 음수가 되는 경우를 막기 위해 `max(0, ...)`를 사용하는 것이 안전합니다.


In [ ]:
def box_area_xyxy(box):
    x1, y1, x2, y2 = box
    width = max(0, x2 - x1)
    height = max(0, y2 - y1)
    return width * height


print('box:', box_xyxy)
print('area:', box_area_xyxy(box_xyxy))


## 10-6. IoU의 의미

IoU는 두 박스가 얼마나 비슷한 위치를 가리키는지 측정하는 값입니다.

$$IoU = \frac{Intersection}{Union}$$

- `Intersection`: 두 박스가 겹치는 영역
- `Union`: 두 박스 중 하나라도 포함하는 전체 영역
- 값의 범위: `0~1`
- `1`에 가까울수록 두 박스가 거의 같은 위치입니다.
- `0`이면 두 박스가 전혀 겹치지 않습니다.

객체 탐지에서는 예측 박스와 정답 박스의 IoU가 일정 기준 이상이면 위치를 맞혔다고 판단합니다. 흔히 `IoU >= 0.5`를 기본 기준으로 사용합니다.


In [ ]:
def intersection_xyxy(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    return (inter_x1, inter_y1, inter_x2, inter_y2)


def iou_xyxy(box_a, box_b):
    inter_box = intersection_xyxy(box_a, box_b)
    inter_area = box_area_xyxy(inter_box)
    area_a = box_area_xyxy(box_a)
    area_b = box_area_xyxy(box_b)
    union_area = area_a + area_b - inter_area

    if union_area == 0:
        return 0.0
    return inter_area / union_area


gt_box = (45, 35, 145, 125)
pred_good = (50, 42, 148, 128)
pred_shifted = (105, 55, 205, 145)
pred_miss = (155, 20, 220, 80)

for name, pred_box in [('good', pred_good), ('shifted', pred_shifted), ('miss', pred_miss)]:
    print(f'{name:8s} IoU = {iou_xyxy(gt_box, pred_box):.3f}')


## 10-7. IoU 시각화

숫자만 보면 직관이 약할 수 있으므로, 정답 박스와 예측 박스가 실제로 어떻게 겹치는지 시각화해 봅니다.


In [ ]:
def draw_box(ax, box, label, color, linewidth=2.5, fill=False, alpha=0.15):
    x1, y1, x2, y2 = box
    ax.add_patch(
        Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=fill,
            edgecolor=color,
            facecolor=color if fill else 'none',
            linewidth=linewidth,
            alpha=alpha if fill else 1.0,
        )
    )
    ax.text(x1, max(10, y1 - 5), label, color=color, fontsize=11, weight='bold')


def visualize_iou(pred_box, title):
    inter_box = intersection_xyxy(gt_box, pred_box)
    iou = iou_xyxy(gt_box, pred_box)

    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.set_xlim(0, image_width)
    ax.set_ylim(image_height, 0)
    ax.set_facecolor('#f8fafc')
    ax.set_title(f'{title} | IoU={iou:.3f}')
    ax.set_xticks([])
    ax.set_yticks([])

    draw_box(ax, gt_box, 'GT', 'crimson')
    draw_box(ax, pred_box, 'Pred', 'royalblue')
    draw_box(ax, inter_box, 'Intersection', 'seagreen', fill=True, alpha=0.25)
    plt.show()


visualize_iou(pred_good, '좋은 예측')
visualize_iou(pred_shifted, '위치가 밀린 예측')
visualize_iou(pred_miss, '거의 빗나간 예측')


## 10-8. IoU threshold로 정답 여부 판단하기

탐지 모델의 예측을 평가할 때는 보통 IoU 기준을 정합니다. 예를 들어 `IoU >= 0.5`이면 위치를 맞힌 것으로 보고, 그보다 작으면 틀린 것으로 볼 수 있습니다.

단, 실제 평가에서는 클래스도 함께 맞아야 합니다. 고양이 위치를 잘 잡았더라도 클래스를 강아지로 예측하면 올바른 탐지가 아닙니다.


In [ ]:
predictions = [
    {'class': 'cat', 'box': pred_good, 'score': 0.92},
    {'class': 'cat', 'box': pred_shifted, 'score': 0.76},
    {'class': 'dog', 'box': pred_good, 'score': 0.81},
    {'class': 'cat', 'box': pred_miss, 'score': 0.60},
]

gt_class = 'cat'
iou_threshold = 0.5

for pred in predictions:
    iou = iou_xyxy(gt_box, pred['box'])
    class_ok = pred['class'] == gt_class
    box_ok = iou >= iou_threshold
    result = 'TP 후보' if class_ok and box_ok else '오답'
    print(
        f"class={pred['class']:<3} score={pred['score']:.2f} "
        f"IoU={iou:.3f} class_ok={class_ok} box_ok={box_ok} -> {result}"
    )


## 10-9. Confidence score 해석

객체 탐지 모델은 보통 각 예측 박스마다 confidence score를 함께 출력합니다. 이 점수는 모델이 해당 박스에 객체가 있고, 그 예측이 믿을 만하다고 보는 정도를 나타냅니다.

실제 모델마다 정의는 조금씩 다르지만, YOLO 계열에서는 대략 다음 정보가 결합됩니다.

- 이 박스 안에 객체가 있을 가능성
- 예측한 클래스가 맞을 가능성
- 박스 위치가 객체와 잘 맞을 가능성

confidence가 높다고 항상 정답은 아닙니다. 위치가 틀렸거나 클래스가 틀렸을 수 있습니다. 그래서 confidence threshold와 IoU 기준을 함께 사용합니다.


In [ ]:
score_threshold = 0.7

print(f'confidence threshold = {score_threshold}')
for pred in predictions:
    keep = pred['score'] >= score_threshold
    print(f"score={pred['score']:.2f}, class={pred['class']:<3}, keep={keep}")


## 10-10. 여러 예측 박스가 겹칠 때

실제 탐지 모델은 같은 객체 주변에 여러 개의 비슷한 박스를 출력할 수 있습니다. 아래 예시처럼 고양이 하나에 대해 score가 높은 박스와 낮은 박스가 동시에 나올 수 있습니다.

이런 중복 박스를 정리하는 대표적인 후처리가 **NMS(Non-Maximum Suppression)** 입니다. NMS는 다음 노트북에서 자세히 다룹니다.


In [ ]:
duplicate_preds = [
    {'box': (48, 38, 146, 126), 'score': 0.95},
    {'box': (52, 42, 150, 130), 'score': 0.88},
    {'box': (60, 50, 155, 135), 'score': 0.73},
    {'box': (150, 25, 220, 90), 'score': 0.64},
]

fig, ax = plt.subplots(figsize=(7, 5))
ax.set_xlim(0, image_width)
ax.set_ylim(image_height, 0)
ax.set_facecolor('#f8fafc')
ax.set_title('같은 객체 주변의 중복 예측 박스')
ax.set_xticks([])
ax.set_yticks([])

draw_box(ax, gt_box, 'GT', 'crimson')
for idx, pred in enumerate(duplicate_preds, start=1):
    label = f"P{idx}: {pred['score']:.2f}"
    draw_box(ax, pred['box'], label, 'royalblue', linewidth=1.8)

plt.show()

for i in range(len(duplicate_preds)):
    for j in range(i + 1, len(duplicate_preds)):
        iou = iou_xyxy(duplicate_preds[i]['box'], duplicate_preds[j]['box'])
        print(f'P{i + 1} vs P{j + 1}: IoU={iou:.3f}')


## 10-11. 직접 실험해 보기

아래 박스 값을 바꿔 보면서 IoU가 어떻게 변하는지 확인해 보세요.

- 예측 박스가 정답 박스와 완전히 같으면 IoU는 `1`입니다.
- 예측 박스가 조금씩 이동하면 IoU가 감소합니다.
- 박스 크기가 너무 커도 union이 커지기 때문에 IoU가 낮아질 수 있습니다.
- 두 박스가 겹치지 않으면 IoU는 `0`입니다.


In [ ]:
my_pred_box = (40, 30, 150, 130)

print('GT box     :', gt_box)
print('My pred box:', my_pred_box)
print('IoU        :', round(iou_xyxy(gt_box, my_pred_box), 3))

visualize_iou(my_pred_box, '직접 입력한 예측 박스')


## 정리

이번 노트북에서 다룬 핵심은 다음과 같습니다.

- Bounding box는 객체 위치를 직사각형 좌표로 표현합니다.
- `xyxy`, `xywh`, `cxcywh`는 같은 박스를 표현하는 서로 다른 방식입니다.
- 정규화 좌표는 이미지 크기와 무관하게 `0~1` 범위로 박스를 표현합니다.
- IoU는 두 박스의 겹침 정도를 `Intersection / Union`으로 계산합니다.
- 객체 탐지 평가는 클래스, confidence score, IoU 기준을 함께 봐야 합니다.
- 실제 모델은 중복 박스를 많이 만들 수 있으므로 다음 단계에서는 NMS가 필요합니다.

다음 노트북 `11_NMS와_탐지_평가.ipynb`에서는 중복 박스를 제거하는 NMS와 precision, recall, mAP 같은 탐지 평가 지표로 이어갑니다.
